# DocIntel RAG Evaluation — RAGAS

Evaluates the production RAG pipeline using [RAGAS](https://docs.ragas.io) metrics.

## Metrics
| Metric | Target | What it measures |
|---|---|---|
| **Context Precision** | >0.6 | Are retrieved chunks actually relevant? |
| **Context Recall** | >0.8 | Did we retrieve all the chunks needed? |
| **Faithfulness** | >0.8 | Is the answer grounded in the retrieved chunks? |
| **Answer Relevancy** | >0.7 | Does the answer address the question? |

## Prerequisites
- DocIntel stack running (`docker compose up`)
- Sample datasets loaded via the admin UI or `/sample-datasets/load` endpoint
- An LLM accessible for RAGAS evaluation (Ollama with `qwen3:4b` by default)

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
RAG_SERVICE_URL  = "http://localhost:8000"   # direct to RAG service
GATEWAY_URL      = "http://localhost:8080"   # via API gateway (needs auth token)
TENANT_ID        = "default"
OLLAMA_BASE_URL  = "http://localhost:11434"
EVAL_MODEL       = "qwen3:4b"               # model used by RAGAS judge calls
N_SAMPLES        = 20                        # questions to evaluate per domain
REQUEST_TIMEOUT  = 120.0                     # seconds

print("Configuration set.")

In [ ]:
import httpx, json, time
from typing import Optional

def query_rag(
    question: str,
    tenant_id: str = TENANT_ID,
    document_type: Optional[str] = None,
    use_cache: bool = False,
) -> dict:
    """Call the RAG service /query endpoint."""
    payload = {
        "question": question,
        "tenant_id": tenant_id,
        "user_roles": [],
        "use_cache": use_cache,
        "use_reranking": True,
    }
    if document_type:
        payload["document_type"] = document_type

    resp = httpx.post(
        f"{RAG_SERVICE_URL}/query",
        json=payload,
        timeout=REQUEST_TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()


# Smoke test
try:
    result = query_rag("What types of documents are available?", use_cache=False)
    print("✓ RAG service reachable")
    print(f"  Answer preview: {result['answer'][:120]}...")
    print(f"  Sources: {len(result['sources'])} | Latency: {result['latency_ms']}ms")
except Exception as e:
    print(f"✗ RAG service not reachable: {e}")
    print("  Make sure docker compose up is running and datasets are loaded.")

## 1. Define Evaluation Questions

Each sample has:
- `question` — the query to send
- `ground_truth` — the expected answer (for Faithfulness / Answer Relevancy)
- `domain` — optional domain filter

In [ ]:
EVAL_QUESTIONS = [
    # ── Technical ──────────────────────────────────────────────────────────
    {
        "question": "What is the difference between REST and GraphQL APIs?",
        "ground_truth": "REST uses fixed endpoints and HTTP verbs; GraphQL uses a single endpoint with flexible queries.",
        "domain": "technical",
    },
    {
        "question": "How does pagination work in REST APIs?",
        "ground_truth": "REST APIs commonly use cursor-based or offset/limit pagination strategies.",
        "domain": "technical",
    },
    {
        "question": "What authentication methods are described in the documentation?",
        "ground_truth": "Bearer tokens, API keys, and OAuth 2.0 flows.",
        "domain": "technical",
    },
    # ── HR Policy ───────────────────────────────────────────────────────────
    {
        "question": "How many days of annual leave are employees entitled to?",
        "ground_truth": "The policy specifies the number of annual leave days per employment year.",
        "domain": "hr_policy",
    },
    {
        "question": "What is the process for requesting remote work?",
        "ground_truth": "Employees submit a remote work request to their manager with advance notice.",
        "domain": "hr_policy",
    },
    {
        "question": "What are the performance review criteria?",
        "ground_truth": "Performance reviews assess goals, competencies, and manager feedback.",
        "domain": "hr_policy",
    },
    # ── Contracts ──────────────────────────────────────────────────────────
    {
        "question": "What are the termination clauses in the SaaS agreement?",
        "ground_truth": "Either party may terminate with 30 days written notice or immediately for material breach.",
        "domain": "contracts",
    },
    {
        "question": "What is the liability limitation in the contract?",
        "ground_truth": "Liability is typically capped at the fees paid in the preceding 12 months.",
        "domain": "contracts",
    },
    # ── Cross-domain ────────────────────────────────────────────────────────
    {
        "question": "What is the company's data retention policy?",
        "ground_truth": "Data retention periods are defined per data type and jurisdiction.",
        "domain": None,
    },
    {
        "question": "How is user access managed?",
        "ground_truth": "Access is controlled via role-based permissions and reviewed periodically.",
        "domain": None,
    },
]

print(f"Defined {len(EVAL_QUESTIONS)} evaluation questions")

## 2. Collect RAG Responses

In [ ]:
import time

eval_samples = []
errors = []

for i, sample in enumerate(EVAL_QUESTIONS):
    print(f"[{i+1}/{len(EVAL_QUESTIONS)}] {sample['question'][:70]}...")
    try:
        result = query_rag(
            question=sample["question"],
            document_type=sample.get("domain"),
            use_cache=False,
        )

        # Build context list from retrieved sources
        contexts = [s["content"] for s in result.get("sources", [])]

        eval_samples.append({
            "question":      sample["question"],
            "answer":        result["answer"],
            "contexts":      contexts,
            "ground_truth":  sample["ground_truth"],
            "domain":        sample.get("domain", "all"),
            "latency_ms":    result["latency_ms"],
            "cache_hit":     result["cache_hit"],
            "source_count":  len(contexts),
        })
        print(f"   ✓ {result['latency_ms']}ms | {len(contexts)} sources | cache={result['cache_hit']}")

    except Exception as e:
        errors.append({"question": sample["question"], "error": str(e)})
        print(f"   ✗ {e}")

    time.sleep(0.5)  # brief pause between requests

print(f"\nCollected {len(eval_samples)} samples, {len(errors)} errors")

In [ ]:
# Quick preview
import pandas as pd

df_raw = pd.DataFrame(eval_samples)
display(df_raw[["question", "domain", "source_count", "latency_ms", "cache_hit"]].head(10))

print(f"\nLatency: p50={df_raw['latency_ms'].median():.0f}ms  p95={df_raw['latency_ms'].quantile(.95):.0f}ms")
print(f"Avg sources retrieved: {df_raw['source_count'].mean():.1f}")

## 3. Run RAGAS Evaluation

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from datasets import Dataset

# Prepare RAGAS Dataset
ragas_data = Dataset.from_dict({
    "question":     [s["question"]    for s in eval_samples],
    "answer":       [s["answer"]      for s in eval_samples],
    "contexts":     [s["contexts"]    for s in eval_samples],
    "ground_truth": [s["ground_truth"] for s in eval_samples],
})

# Use local Ollama for RAGAS judge calls (avoids OpenAI dependency)
ollama_llm = LangchainLLMWrapper(
    ChatOllama(base_url=OLLAMA_BASE_URL, model=EVAL_MODEL)
)
ollama_embed = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(base_url=OLLAMA_BASE_URL, model="nomic-embed-text")
)

metrics = [context_precision, context_recall, faithfulness, answer_relevancy]
for m in metrics:
    m.llm = ollama_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ollama_embed

print("Running RAGAS evaluation (this may take several minutes)...")
ragas_result = evaluate(ragas_data, metrics=metrics)
print("Done.")
print(ragas_result)

## 4. Results

In [ ]:
import pandas as pd

TARGETS = {
    "context_precision": 0.60,
    "context_recall":    0.80,
    "faithfulness":      0.80,
    "answer_relevancy":  0.70,
}

scores = {
    "context_precision": ragas_result["context_precision"],
    "context_recall":    ragas_result["context_recall"],
    "faithfulness":      ragas_result["faithfulness"],
    "answer_relevancy":  ragas_result["answer_relevancy"],
}

rows = []
for metric, score in scores.items():
    target = TARGETS[metric]
    status = "✓ PASS" if score >= target else "✗ FAIL"
    rows.append({"metric": metric, "score": round(score, 3), "target": target, "status": status})

df_scores = pd.DataFrame(rows)
print("\n=== RAGAS Evaluation Results ===")
print(df_scores.to_string(index=False))

In [ ]:
# Per-sample breakdown
df_detail = ragas_result.to_pandas()
df_detail["question_short"] = df_detail["question"].str[:60]
display(df_detail[["question_short", "context_precision", "context_recall",
                    "faithfulness", "answer_relevancy"]].round(3))

In [ ]:
# Domain breakdown
df_detail["domain"] = [s["domain"] for s in eval_samples]
print("\n=== Scores by Domain ===")
print(
    df_detail
    .groupby("domain")[["context_precision", "context_recall", "faithfulness", "answer_relevancy"]]
    .mean()
    .round(3)
    .to_string()
)

## 5. Retrieval Metrics (Recall@K, MRR)

In [ ]:
def reciprocal_rank(sources: list[dict], relevant_keywords: list[str]) -> float:
    """Compute MRR for a single query given keyword-based relevance heuristic."""
    for rank, src in enumerate(sources, start=1):
        content = src.get("content", "").lower()
        if any(kw.lower() in content for kw in relevant_keywords):
            return 1.0 / rank
    return 0.0


KEYWORD_RELEVANCE = {
    "What is the difference between REST and GraphQL APIs?": ["graphql", "rest", "endpoint"],
    "How many days of annual leave are employees entitled to?": ["leave", "annual", "days", "vacation"],
    "What are the termination clauses in the SaaS agreement?": ["termination", "notice", "breach"],
}

mrr_scores = []
for s in eval_samples:
    keywords = KEYWORD_RELEVANCE.get(s["question"])
    if keywords and s["contexts"]:
        # Reconstruct dummy sources from stored context strings
        dummy = [{"content": c} for c in s["contexts"]]
        rr = reciprocal_rank(dummy, keywords)
        mrr_scores.append(rr)

if mrr_scores:
    print(f"MRR (subset with keyword annotations): {sum(mrr_scores)/len(mrr_scores):.3f}")
else:
    print("No keyword annotations available for MRR calculation.")

# Recall@K based on source_count
avg_sources = df_raw["source_count"].mean()
empty_pct   = (df_raw["source_count"] == 0).mean() * 100
print(f"\nRetrieval stats:")
print(f"  Avg sources per query : {avg_sources:.1f}")
print(f"  Queries with 0 sources: {empty_pct:.1f}%  ← should be <5%")

## 6. Latency Analysis

In [ ]:
latencies = df_raw["latency_ms"]

print("=== Latency Analysis ===")
print(f"  p50 : {latencies.quantile(.50):.0f}ms  (target: <2000ms)")
print(f"  p95 : {latencies.quantile(.95):.0f}ms  (target: <5000ms)")
print(f"  p99 : {latencies.quantile(.99):.0f}ms")
print(f"  max : {latencies.max():.0f}ms")
print(f"  cache hit rate: {df_raw['cache_hit'].mean()*100:.0f}%")

## 7. Save Results

In [ ]:
import json
from datetime import datetime

report = {
    "timestamp": datetime.utcnow().isoformat(),
    "n_samples": len(eval_samples),
    "ragas_scores": scores,
    "targets": TARGETS,
    "latency": {
        "p50_ms":  int(latencies.quantile(.50)),
        "p95_ms":  int(latencies.quantile(.95)),
        "max_ms":  int(latencies.max()),
    },
    "retrieval": {
        "avg_sources": round(float(avg_sources), 2),
        "empty_pct":   round(float(empty_pct), 1),
    },
}

report_path = f"eval_report_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved → {report_path}")
print(json.dumps(report, indent=2))